# CatalystIQ Data Exploration - End to End

Single notebook workflow for Polygon-first profiling, Finnhub parity check, and scanner threshold recommendations.

## 0) Setup

- Ensure `data-exploration/.env` contains API keys.
- Run cells top-to-bottom.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path('..').resolve()))

from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
import pandas as pd
import plotly.express as px

from src.common import (
    load_env, get_api_keys, load_config, new_pull_id,
    polygon_gainers, polygon_snapshot_all, polygon_intraday_bars,
    finnhub_quote, normalize_polygon_snapshot, normalize_finnhub_quote,
    quality_summary, save_raw_json, save_parquet
)

load_env('../.env')
cfg = load_config('../config/exploration.yaml')
keys = get_api_keys()

assert keys['polygon'], 'POLYGON_API_KEY missing in data-exploration/.env'


## 1) Polygon Snapshot Profile

In [ ]:
pull_snapshot = new_pull_id('polygon_snapshot')

gainers_payload = polygon_gainers(keys['polygon'], keys['polygon_base'])
all_payload = polygon_snapshot_all(keys['polygon'], keys['polygon_base'])

save_raw_json(gainers_payload, cfg.outputs_raw_dir / f'{pull_snapshot}_gainers.json')
save_raw_json(all_payload, cfg.outputs_raw_dir / f'{pull_snapshot}_all_tickers.json')

df_gainers = normalize_polygon_snapshot(gainers_payload, '/v2/snapshot/locale/us/markets/stocks/gainers', pull_snapshot)
df_all = normalize_polygon_snapshot(all_payload, '/v2/snapshot/locale/us/markets/stocks/tickers', pull_snapshot)

df_polygon = pd.concat([df_gainers, df_all], ignore_index=True).drop_duplicates(
    subset=['provider', 'symbol', 'ts_utc', 'source_endpoint']
)

save_parquet(df_polygon, cfg.outputs_processed_dir / f'{pull_snapshot}_polygon_snapshot.parquet')
summary_polygon = quality_summary(df_polygon)
summary_polygon


In [ ]:
fig_cp = px.histogram(df_polygon, x='change_pct', nbins=80, title='Polygon Change % Distribution')
fig_cp.show()

fig_vol = px.histogram(df_polygon, x='volume', nbins=80, title='Polygon Volume Distribution (log x)')
fig_vol.update_xaxes(type='log')
fig_vol.show()


## 2) Polygon 2-Day Intraday Profile (Top Movers)

In [ ]:
top_payload = polygon_gainers(keys['polygon'], keys['polygon_base'])
tickers = [x.get('ticker') for x in top_payload.get('tickers', []) if x.get('ticker')][:25]

end_date = datetime.now(tz=ZoneInfo('UTC')).date()
start_date = (end_date - timedelta(days=4))

pull_intraday = new_pull_id('polygon_intraday')
rows = []

for symbol in tickers:
    bars = polygon_intraday_bars(keys['polygon'], keys['polygon_base'], symbol, start_date.isoformat(), end_date.isoformat())
    save_raw_json(bars, cfg.outputs_raw_dir / f'{pull_intraday}_{symbol}_bars.json')
    for r in bars.get('results', []) or []:
        rows.append({
            'provider': 'polygon',
            'symbol': symbol,
            'ts_utc': pd.to_datetime(r['t'], unit='ms', utc=True),
            'open': r.get('o'),
            'high': r.get('h'),
            'low': r.get('l'),
            'close': r.get('c'),
            'volume': r.get('v'),
            'source_endpoint': '/v2/aggs/ticker/{symbol}/range/1/minute/{from}/{to}',
            'pull_id': pull_intraday,
        })

df_intraday = pd.DataFrame(rows).sort_values(['symbol', 'ts_utc'])
if not df_intraday.empty:
    df_intraday['ret_1m_pct'] = df_intraday.groupby('symbol')['close'].pct_change() * 100
    df_intraday['rv_20m'] = df_intraday.groupby('symbol')['ret_1m_pct'].rolling(20).std().reset_index(level=0, drop=True)

save_parquet(df_intraday, cfg.outputs_processed_dir / f'{pull_intraday}_polygon_intraday_2d.parquet')
df_intraday.head()


In [ ]:
if not df_intraday.empty:
    intraday_profile = df_intraday.groupby('symbol').agg(
        bars=('symbol', 'size'),
        avg_volume=('volume', 'mean'),
        p95_abs_ret=('ret_1m_pct', lambda s: s.abs().quantile(0.95)),
        avg_rv_20m=('rv_20m', 'mean')
    ).sort_values('p95_abs_ret', ascending=False)
    display(intraday_profile.head(20))

    fig_box = px.box(df_intraday, x='symbol', y='ret_1m_pct', title='1-Min Return Distribution by Symbol')
    fig_box.show()


## 3) Finnhub Parity Check

In [ ]:
if not keys['finnhub']:
    print('FINNHUB_API_KEY missing. Skipping parity section.')
    df_cmp = pd.DataFrame()
else:
    pull_parity = new_pull_id('parity')
    poly_parity_payload = polygon_gainers(keys['polygon'], keys['polygon_base'])
    save_raw_json(poly_parity_payload, cfg.outputs_raw_dir / f'{pull_parity}_polygon_gainers.json')

    df_poly_parity = normalize_polygon_snapshot(poly_parity_payload, '/v2/snapshot/locale/us/markets/stocks/gainers', pull_parity)
    parity_symbols = df_poly_parity['symbol'].dropna().unique().tolist()[:25]

    fh_frames = []
    for symbol in parity_symbols:
        q = finnhub_quote(keys['finnhub'], keys['finnhub_base'], symbol)
        save_raw_json(q, cfg.outputs_raw_dir / f'{pull_parity}_{symbol}_finnhub_quote.json')
        fh_frames.append(normalize_finnhub_quote(symbol, q, '/quote', pull_parity))

    df_fh = pd.concat(fh_frames, ignore_index=True) if fh_frames else pd.DataFrame()

    df_cmp = df_poly_parity[['symbol','last_price','change_pct','volume','prev_close']].merge(
        df_fh[['symbol','last_price','change_pct','volume','prev_close']],
        on='symbol', how='inner', suffixes=('_polygon','_finnhub')
    )

    if not df_cmp.empty:
        df_cmp['last_price_diff_pct'] = ((df_cmp['last_price_finnhub'] - df_cmp['last_price_polygon']) / df_cmp['last_price_polygon']) * 100
        df_cmp['change_pct_diff'] = df_cmp['change_pct_finnhub'] - df_cmp['change_pct_polygon']

    save_parquet(df_cmp, cfg.outputs_processed_dir / f'{pull_parity}_provider_parity.parquet')

df_cmp.head()


In [ ]:
if not df_cmp.empty:
    parity_stats = df_cmp[['last_price_diff_pct','change_pct_diff']].describe(percentiles=[0.5, 0.9, 0.95]).T
    display(parity_stats)


## 4) Scanner Threshold Recommendations

In [ ]:
df_thresh = df_polygon.dropna(subset=['change_pct','volume']).copy()

pct_candidates = [2, 3, 4, 5, 6, 8, 10]
vol_candidates = [50000, 100000, 200000, 500000, 1000000]

grid_rows = []
for p in pct_candidates:
    for v in vol_candidates:
        matched = df_thresh[(df_thresh['change_pct'] >= p) & (df_thresh['volume'] >= v)]
        grid_rows.append({
            'min_change_pct': p,
            'min_volume': v,
            'match_count': len(matched),
            'median_change_pct': matched['change_pct'].median() if not matched.empty else None,
            'median_volume': matched['volume'].median() if not matched.empty else None,
        })

df_grid = pd.DataFrame(grid_rows)
display(df_grid.sort_values(['min_change_pct','min_volume']).head(40))

pivot = df_grid.pivot(index='min_volume', columns='min_change_pct', values='match_count')
fig_heat = px.imshow(pivot, text_auto=True, title='Threshold Match Count Heatmap')
fig_heat.update_layout(xaxis_title='min_change_pct', yaxis_title='min_volume')
fig_heat.show()

baseline = df_grid[(df_grid['min_change_pct'] == 4) & (df_grid['min_volume'] == 100000)]
print('Baseline recommendation')
display(baseline)


## 5) Write Findings to Markdown Report

Fill `reports/price_volume_exploration.md` with outputs from this run (coverage, quality, parity, and threshold recs).